# 08 Run Static Allocator Baseline v1

This notebook is a thin execution wrapper around `run_static_allocator_baseline_v1.py`. The production logic lives in the Python script; this notebook only sets parameters, runs the script, and previews the generated outputs.

Run this notebook from the existing `portfolio_allocation_rl` conda environment.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path('/Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl').resolve()
SCRIPT_PATH = PROJECT_ROOT / 'run_static_allocator_baseline_v1.py'

assert PROJECT_ROOT.exists(), f'Project root not found: {PROJECT_ROOT}'
assert SCRIPT_PATH.exists(), f'Allocator script not found: {SCRIPT_PATH}'

PROJECT_ROOT, SCRIPT_PATH

## Configuration

Edit these values to run alternative baseline settings, such as 20 bps transaction costs or a max-weight cap.

In [ ]:
config = {
    'project_root': PROJECT_ROOT,
    'pred_file': 'data/prediction/fm_oos_predictions.parquet',
    'risk_dir': 'data/risk/risk_cov_npz',
    'risk_meta_file': 'data/risk/risk_monthly_metadata.csv',
    'returns_file': 'data/panel/monthly_stock_panel_basic_full.parquet',
    'outdir': 'data/allocator',
    'lambda_risk': 10.0,
    'tau_turnover': 0.001,
    'cost_bps': 10.0,
    'solver': 'CLARABEL',
    'max_weight': None,
    'start_month': None,
    'end_month': None,
    'warm_start': 'true',
    'log_level': 'INFO',
}

config

## Run Allocator Script

In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    '--project-root', str(config['project_root']),
    '--pred-file', config['pred_file'],
    '--risk-dir', config['risk_dir'],
    '--risk-meta-file', config['risk_meta_file'],
    '--returns-file', config['returns_file'],
    '--outdir', config['outdir'],
    '--lambda-risk', str(config['lambda_risk']),
    '--tau-turnover', str(config['tau_turnover']),
    '--cost-bps', str(config['cost_bps']),
    '--solver', config['solver'],
    '--warm-start', config['warm_start'],
    '--log-level', config['log_level'],
]

if config['max_weight'] is not None:
    cmd.extend(['--max-weight', str(config['max_weight'])])
if config['start_month'] is not None:
    cmd.extend(['--start-month', str(config['start_month'])])
if config['end_month'] is not None:
    cmd.extend(['--end-month', str(config['end_month'])])

print(' '.join(cmd))
result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)

print(result.stdout)
if result.stderr:
    print(result.stderr)

result.check_returncode()

## Output Paths

In [ ]:
outdir = PROJECT_ROOT / config['outdir']

weights_path = outdir / 'static_allocator_weights.parquet'
backtest_path = outdir / 'static_allocator_backtest.csv'
summary_path = outdir / 'static_allocator_summary.txt'
cumret_plot = outdir / 'static_allocator_cumret.png'
turnover_plot = outdir / 'static_allocator_turnover.png'
n_assets_plot = outdir / 'static_allocator_n_assets.png'

for path in [weights_path, backtest_path, summary_path, cumret_plot, turnover_plot, n_assets_plot]:
    print(path, 'exists=' + str(path.exists()))

## Preview Backtest Results

In [ ]:
backtest = pd.read_csv(backtest_path, parse_dates=['month_end'])
backtest.head()

In [ ]:
backtest.tail()

## Summary

In [ ]:
print(summary_path.read_text())